In [ ]:
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 3 Lab — Feature Engineering and the ML Pipeline
**Introduction to Data Science — DBAS 5015**

---

<!-- ============================================================
  WRITING STYLE — apply to all markdown cells in this notebook
  - Direct and to the point. Say what the concept is and move on.
  - No hedging. State things plainly.
  - Active voice. 'The function returns a value' not 'A value is returned.'
  - Short sentences for instructions. Students need to scan them quickly.
  - Exercise instructions must be unambiguous — one reading, one meaning.
  - No filler. Every sentence should add information or cut it.
  ============================================================ -->

### What This Lab Covers
- Creating the target variable
- One-hot encoding with `pd.get_dummies` and `OneHotEncoder`
- Scaling numeric features with `StandardScaler`
- Splitting into training and test sets — in the correct order
- Combining preprocessing with `ColumnTransformer`

**Estimated time:** 75–90 minutes

---

## The Dataset

This lab uses a pre-cleaned version of the orders dataset from Week 2. Run the cell below to load it — all type problems, invalid values, and duplicates have already been resolved.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

np.random.seed(7)
n = 285  # post-cleaning size from Week 2

regions   = np.random.choice(['Northeast', 'West', 'South', 'Midwest'], n)
products  = np.random.choice(['Laptop Stand', 'Monitor Arm', 'Keyboard Tray', 'Cable Manager'], n)
units     = np.random.randint(0, 25, n)
price     = np.random.choice([49.95, 79.95, 34.95, 24.95], n)
discount  = np.clip(np.random.choice([0, 0.05, 0.10, 0.15, 0.20], n), 0, 1)
month     = np.random.randint(1, 13, n)
quarter   = ((month - 1) // 3) + 1
revenue   = np.round(units * price * (1 - discount), 2)
rev_miss  = np.zeros(n, dtype=int)

df = pd.DataFrame({
    'region':          regions,
    'product':         products,
    'units_sold':      units,
    'unit_price':      price,
    'discount_pct':    discount,
    'order_month':     month,
    'order_quarter':   quarter,
    'revenue':         revenue,
    'revenue_missing': rev_miss,
})

print(f"Dataset ready: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.dtypes)

---
## Part 1 — Creating the Target Variable

Before encoding or splitting, define what you are predicting. This lab uses a binary target: 1 if a transaction's revenue is above the dataset median, 0 if not. This is a classification problem — predicting which transactions are high-revenue.

In [ ]:
# Create the binary target
median_revenue = df['revenue'].median()
df['high_revenue'] = (df['revenue'] > median_revenue).astype(int)

print(f"Revenue median: ${median_revenue:.2f}")
print(f"\nTarget distribution:")
print(df['high_revenue'].value_counts())
print(f"\nClass balance: {df['high_revenue'].mean():.1%} positive")

---
### 💼 Business Context — Target Definition Is a Business Decision

The choice to use median as the threshold is arbitrary here — it ensures a balanced dataset, which is convenient for learning. In practice, the threshold would be defined by the business question: a "high-value transaction" might be one above $500, or in the top 20% of revenue, or above the cost-to-acquire threshold. The model learns whatever pattern you define as the target. Define it wrong and the model optimizes for the wrong thing.

---

### ✏️ Exercise 1.1 — Separate Features and Target

1. Separate the dataset into `X` (features) and `y` (target). `X` should contain all columns except `revenue` and `high_revenue`. `y` should be the `high_revenue` column.
2. Print the shape of `X` and the value counts of `y`.
3. List the categorical columns and numeric columns in `X` as two Python lists — you will need them in Part 4.

In [ ]:
# Exercise 1.1

# 1. Separate features and target
X = 
y = 

# 2. Shapes and target distribution
print("X shape:", X.shape)
print("y value counts:")
print(y.value_counts())

# 3. Column type lists (fill these in — used in Part 4)
categorical_cols = []  # string columns
numerical_cols   = []  # numeric columns

print("\nCategorical:", categorical_cols)
print("Numerical:  ", numerical_cols)

---
## Part 2 — One-Hot Encoding

Two categorical columns need encoding: `region` (4 categories) and `product` (4 categories). This part demonstrates both the pandas and scikit-learn approaches side by side.

| Approach | When to Use |
|:---------|:------------|
| `pd.get_dummies` | Quick exploration and EDA |
| `OneHotEncoder` | Inside a scikit-learn pipeline — required for production |

In [ ]:
# Pandas approach — get_dummies
X_dummies = pd.get_dummies(X, columns=['region', 'product'], drop_first=True)

print("Columns after get_dummies:")
print(X_dummies.columns.tolist())
print(f"\nShape: {X_dummies.shape}")
print(f"\nSample rows:")
print(X_dummies.head(3))

In [ ]:
# scikit-learn approach — OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

encoded_array = ohe.fit_transform(X[['region', 'product']])
encoded_cols  = ohe.get_feature_names_out(['region', 'product'])

print("Encoded column names:")
print(encoded_cols)
print(f"\nEncoded array shape: {encoded_array.shape}")
print(f"\nFirst 3 rows:")
print(encoded_array[:3])

---
### 🤖 ML Connection — Why `get_dummies` Is Not Enough

`pd.get_dummies` fits and transforms in one step — you cannot call `.transform()` separately on new data. This means you cannot apply the same encoding to unseen test data without re-fitting. `OneHotEncoder` separates fitting from transforming, which is what `Pipeline` and `ColumnTransformer` require. In exploration, `get_dummies` is fast and readable. In any pipeline that will be applied to new data, `OneHotEncoder` is the only correct choice.

---

### ✏️ Exercise 2.1 — Ordinal Encoding

The dataset does not have an ordinal column, so you will create one. Add a `priority` column to `X` with values `'Low'`, `'Medium'`, and `'High'` assigned randomly.

1. Add `priority` to `X` using `np.random.choice` with `random_state` already set.
2. Apply `OrdinalEncoder` with the correct category order: `['Low', 'Medium', 'High']`.
3. Print the first 10 original values alongside their encoded values to verify the mapping is correct.
4. Explain in a comment why one-hot encoding `priority` would be the wrong choice.

In [ ]:
# Exercise 2.1 — ordinal encoding
np.random.seed(99)

# 1. Add priority column
X = X.copy()
X['priority'] = np.random.choice(['Low', 'Medium', 'High'], len(X))

# 2. Apply OrdinalEncoder with correct order
oe = OrdinalEncoder(categories=[
    # your category list here
])
priority_encoded = 

# 3. Print first 10 to verify mapping
check = pd.DataFrame({
    'original': X['priority'].values[:10],
    'encoded':  priority_encoded[:10, 0]
})
print(check)

# 4. Why not one-hot encode priority?
# 

---
## Part 3 — Scaling Numeric Features

The numeric columns in `X` operate on very different scales. `units_sold` ranges from 0 to 24. `unit_price` ranges from $24.95 to $79.95. `discount_pct` ranges from 0 to 0.20. Without scaling, models that use distance or weights will treat `unit_price` as the most important feature simply because its values are largest.

This part demonstrates the effect of scaling and the correct way to apply it.

In [ ]:
# Before scaling — see the range differences
num_cols = ['units_sold', 'unit_price', 'discount_pct', 'order_month', 'order_quarter', 'revenue_missing']
print("Before scaling:")
print(X[num_cols].describe().loc[['min', 'max', 'mean', 'std']].round(3))

In [ ]:
# StandardScaler — fit on the data, transform it
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[num_cols])

# Convert back to DataFrame for readability
X_scaled_df = pd.DataFrame(X_scaled, columns=num_cols)

print("After StandardScaler:")
print(X_scaled_df.describe().loc[['min', 'max', 'mean', 'std']].round(3))
print("\nAll features now have mean ≈ 0 and std ≈ 1")

In [ ]:
# MinMaxScaler — for comparison
mm_scaler = MinMaxScaler()
X_mm = mm_scaler.fit_transform(X[num_cols])
X_mm_df = pd.DataFrame(X_mm, columns=num_cols)

print("After MinMaxScaler:")
print(X_mm_df.describe().loc[['min', 'max', 'mean', 'std']].round(3))
print("\nAll features now fall between 0 and 1")

---
### 🤖 ML Connection — Scaling Is Learned, Not Fixed

The scaler stores the training mean and standard deviation when you call `.fit()`. When you call `.transform()` on test data, it applies those stored values — not the test set's own statistics. This is why `.fit_transform()` is only called on training data. Test data is always transformed using the training set's parameters. The next exercise makes this distinction concrete.

---

### ✏️ Exercise 3.1 — Correct vs. Incorrect Scaling Order

This exercise demonstrates why the split must happen before the scaler is fit.

1. Split `X[num_cols]` and `y` into 80/20 train/test sets with `random_state=42`.
2. **Correct version:** Fit `StandardScaler` on `X_train` only, transform both `X_train` and `X_test`.
3. **Incorrect version:** Fit `StandardScaler` on all of `X[num_cols]` first, then split and transform.
4. Compare the test set means from both versions. They will differ — print both and write a comment explaining what the difference reveals.

In [ ]:
# Exercise 3.1 — correct vs. incorrect scaling order

# 1. Split first
X_num = X[num_cols]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    # your code here
)

# 2. CORRECT — fit on training data only
scaler_correct = StandardScaler()
X_train_correct = 
X_test_correct  = 

# 3. INCORRECT — fit on all data before splitting
scaler_wrong = StandardScaler()
# your code here — fit on X_num, then split

# 4. Compare test set means
print("Test set column means — CORRECT version:")
print(pd.DataFrame(X_test_correct, columns=num_cols).mean().round(4))

print("\nTest set column means — INCORRECT version:")
print(pd.DataFrame(X_test_wrong, columns=num_cols).mean().round(4))

# Comment: what does the difference reveal?
# 

---
## Part 4 — ColumnTransformer: One Object, Two Transformations

`ColumnTransformer` applies different preprocessing to different column subsets in a single, reproducible step. It returns a numpy array with all columns combined — categorical (encoded) columns first, then numeric (scaled) columns.

In [ ]:
# Use the lists defined in Exercise 1.1
# If you did not fill them in, set them here:
categorical_cols = ['region', 'product']
numerical_cols   = ['units_sold', 'unit_price', 'discount_pct',
                    'order_month', 'order_quarter', 'revenue_missing']

# Build the preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_cols),
    ('num', StandardScaler(), numerical_cols),
])

# Split — must happen before fitting the preprocessor
X_model = X[categorical_cols + numerical_cols]   # only the columns we want
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y, test_size=0.2, random_state=42, stratify=y
)

# Fit on training data only, transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"Training set: {X_train_processed.shape}")
print(f"Test set:     {X_test_processed.shape}")
print(f"\nClass balance in y_train: {y_train.mean():.1%} positive")
print(f"Class balance in y_test:  {y_test.mean():.1%} positive")

### ✏️ Exercise 4.1 — Inspect and Verify the Preprocessed Output

The output of `ColumnTransformer` is a numpy array. You need to know what each column represents before passing it to a model.

1. Retrieve the full list of output column names from `preprocessor` using `get_feature_names_out()`.
2. Convert `X_train_processed` to a DataFrame with those column names.
3. Verify: (a) the mean of each scaled numeric column is close to 0, (b) the one-hot columns contain only 0s and 1s.
4. Print the value counts of `y_train` and `y_test` and confirm stratification worked — both should show approximately 50/50.

In [ ]:
# Exercise 4.1

# 1. Get output column names
output_cols = 
print("Output columns:")
print(output_cols)

# 2. Convert to DataFrame
train_df = 

# 3a. Mean of scaled numeric columns — should be close to 0
print("\nMeans of numeric columns (expect ≈ 0):")
print(train_df[numerical_cols].mean().round(4))

# 3b. Unique values in one-hot columns — should be only 0 and 1
ohe_cols = [c for c in output_cols if c.startswith('cat__')]
unique_vals = train_df[ohe_cols].stack().unique()
print(f"\nOne-hot unique values: {sorted(unique_vals)}  (expect [0.0, 1.0])")

# 4. Stratification check
print(f"\ny_train class balance: {y_train.mean():.1%}")
print(f"y_test  class balance: {y_test.mean():.1%}")

---
## Challenge Exercise

This section is optional — attempt it if you finish early or want a stretch.

The preprocessing in this lab treats all numeric columns identically (StandardScaler) and all categorical columns identically (OneHotEncoder). Real datasets often require different treatment for different columns.

Build a `ColumnTransformer` with the following column-specific preprocessing:

| Column | Preprocessing | Reason |
|:-------|:-------------|:-------|
| `region` | `OneHotEncoder(drop='first')` | Nominal, 4 categories |
| `product` | `OneHotEncoder(drop='first')` | Nominal, 4 categories |
| `units_sold` | `StandardScaler` | Continuous, roughly symmetric |
| `unit_price` | `'passthrough'` | Already on a natural scale, only 4 values — no benefit to scaling |
| `discount_pct` | `MinMaxScaler` | Already between 0 and 1 — StandardScaler would expand this range |
| `order_month` | `StandardScaler` | Continuous numeric |
| `order_quarter` | `'passthrough'` | Integer 1–4, already interpretable |
| `revenue_missing` | `'passthrough'` | Binary flag — scaling a 0/1 column destroys its meaning |

1. Build and fit this custom `ColumnTransformer`.
2. Print the output shape and column names.
3. Verify `discount_pct` values fall between 0 and 1 in the processed output.
4. **Reflection:** `unit_price` takes only four values (24.95, 34.95, 49.95, 79.95). Would one-hot encoding it be better than passthrough? What are the trade-offs?

In [ ]:
# Challenge — custom ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

custom_preprocessor = ColumnTransformer(transformers=[
    # ('name', transformer, [columns]),
])

# Fit and transform


# Verify


---
## Week 3 Summary

| Concept | Key Takeaway |
|:--------|:-------------|
| Target variable | Define it as a business decision — what you predict shapes what the model learns |
| One-hot encoding | Creates k-1 binary columns — `drop='first'` avoids the dummy variable trap |
| `handle_unknown='ignore'` | Required for production — outputs all zeros for unseen categories |
| Ordinal encoding | For ordered categories only — never use on nominal variables |
| `get_dummies` vs `OneHotEncoder` | `get_dummies` for exploration; `OneHotEncoder` for pipelines |
| StandardScaler | Mean = 0, std = 1 — correct default for most algorithms |
| MinMaxScaler | Range [0, 1] — use when values must be bounded |
| `'passthrough'` | Keeps a column unchanged inside `ColumnTransformer` |
| Split before fitting | Always split first — fit encoders and scalers on training data only |
| `stratify=y` | Preserves class proportions in both sets — always use for classification |
| `ColumnTransformer` | Applies different preprocessing to different columns in one reproducible step |

---
**Next week:** The first model — linear regression for predicting continuous outcomes. You will use the preprocessed data built this week to train, evaluate, and interpret a regression model.